# Modelverbeteringen

**LET OP: dit is DUMMY data, de scores zeggen niets over echte modellen.**

De trainbare modellen stap voor stap verbeteren, van simpel naar ingewikkeld. Alles wordt alleen op de 5 CV-folds uit het splitmanifest geëvalueerd. De testfamilies blijven dicht.

De tekst en de uitleg per stap staan in `docs/modelverbeteringen.md`. De tabellen daar komen letterlijk uit de output van dit notebook.

## Voorbereiding

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, f1_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.tree import DecisionTreeClassifier

# De hoofdmap van de repo zoeken (daar staat pyproject.toml) en in sys.path zetten.
HOOFDMAP = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(HOOFDMAP))
DATA_MAP = HOOFDMAP / "outputs" / "candidate"

from src.candidate.data.features import selecteer_features
from src.candidate.data.labelling_rule import (
    CATEGORIE_VOORKEUR,
    J_PER_KWH,
    bepaal_label,
    laad_benchmark,
    maak_configuratietabel,
    voldoende_categorieen,
)

# Vaste seed voor alle modellen.
RANDOM_STATE = 42

# De klassen, in de volgorde van de confusion matrix.
LABELS = CATEGORIE_VOORKEUR + ["geen"]

# De middelste folds en de randfolds, net als in train_candidate_selector.py: afgeleid uit
# alleen de folds in het splitmanifest. Op de randfolds moet een model extrapoleren.
with open(DATA_MAP / "split_manifest.json", encoding="utf-8") as bestand:
    foldlengtes = sorted(fold["input_length_tokens"] for fold in json.load(bestand)["cv_folds"])
MIDDEN, RAND = foldlengtes[1:-1], [foldlengtes[0], foldlengtes[-1]]

# Beslisregel voor "blijft een stap staan?", vastgelegd voordat er resultaten zijn.
# Macro-F1 telt pas als hoger bij meer dan 0,005 verschil. Daaronder is hij ongeveer gelijk.
MARGE_F1 = 0.005
# De violation rate is duidelijk hoger bij meer dan 1 procentpunt.
MARGE_VIOLATION = 0.01

# Getal voor "geen budget": ruimer dan elk budget in de data.
GROOT = 1e12

# Drempels voor stap 8 en stap 10.
DREMPELS = [0.5, 0.6, 0.7, 0.8, 0.9]

In [2]:
scenarios = pd.read_csv(DATA_MAP / "scenarios.csv")
configuraties = maak_configuratietabel(laad_benchmark(DATA_MAP / "dummy_benchmark.csv"))
configuraties["family_id"] = (
    configuraties["hardware"] + "__" + configuraties["input_length_tokens"].astype(str)
)

with open(DATA_MAP / "split_manifest.json", encoding="utf-8") as bestand:
    manifest = json.load(bestand)

# De testfamilies gaan er hier uit. Daarna komen ze nergens meer voor.
test_families = manifest["test_families"]
train = scenarios[~scenarios["family_id"].isin(test_families)].reset_index(drop=True)
train_configuraties = configuraties[~configuraties["family_id"].isin(test_families)]
assert not set(train["family_id"]) & set(test_families)

# X is hier nog ruw: de bewerking gebeurt in de pipeline.
x_ruw, y = selecteer_features(train)

# Per scenario de categorieën die de eisen halen, één keer berekend. Dat scheelt tijd,
# want scores() gebruikt dit heel vaak. De definitie is dezelfde als in 01_baseline.ipynb.
voldoende = pd.Series(
    [voldoende_categorieen(configuraties, s) for s in train.to_dict("records")],
    index=train.index,
)

print(len(train), "scenario's,", len(train_configuraties), "configuraties")

1492 scenario's, 72 configuraties


## De maten

`scores()` is de enige functie voor de maten, bij elke stap. De definities zijn dezelfde als in `01_baseline.ipynb`.

In [3]:
def scores(voorspeld, masker=None):
    if masker is None:
        masker = pd.Series(True, index=train.index)

    echt = train.loc[masker, "label"]
    keuze = voorspeld[masker]
    passend = voldoende[masker]

    matrix = pd.DataFrame(
        confusion_matrix(echt, keuze, labels=LABELS), index=LABELS, columns=LABELS
    )

    # Violations: haalt de gekozen categorie de eisen niet?
    gekozen = keuze != "geen"
    violations = sum(k not in p for k, p in zip(keuze[gekozen], passend[gekozen]))

    # Gemiste kansen: "geen" voorspeld terwijl er wel een categorie voldoet.
    kansen = echt != "geen"
    gemist = int((keuze[kansen] == "geen").sum())

    return {
        "macro_f1": f1_score(echt, keuze, labels=LABELS, average="macro", zero_division=0),
        "matrix": matrix,
        "violations": violations,
        "keuzes": int(gekozen.sum()),
        "violation_rate": violations / gekozen.sum() if gekozen.sum() > 0 else 0.0,
        "gemist": gemist,
        "kansen": int(kansen.sum()),
        "gemiste_kans_rate": gemist / kansen.sum() if kansen.sum() > 0 else 0.0,
    }


def masker_lengtes(lengtes):
    return train["input_length_tokens"].isin(lengtes)


def scores_midden(voorspeld):
    return scores(voorspeld, masker_lengtes(MIDDEN))

## Output voor het document

De helpers hieronder printen Markdown-tabellen. Het document neemt die letterlijk over, tussen de markeringen `@@naam` en `@@einde`.

In [4]:
def getal(waarde, cijfers=3):
    return f"{waarde:.{cijfers}f}".replace(".", ",")


def procent(waarde):
    return getal(100 * waarde, 1) + "%"


def blok(naam, tekst):
    print(f"@@{naam}\n{tekst}\n@@einde\n")


def md_tabel(rijen, kolommen):
    regels = ["| " + " | ".join(kolommen) + " |", "|" + "---|" * len(kolommen)]
    for rij in rijen:
        regels.append("| " + " | ".join(str(cel) for cel in rij) + " |")
    return "\n".join(regels)


def resultaat_rij(model, deel, s):
    return [
        model, deel, getal(s["macro_f1"]),
        f"{s['violations']} van {s['keuzes']}", procent(s["violation_rate"]),
        f"{s['gemist']} van {s['kansen']}", procent(s["gemiste_kans_rate"]),
    ]


def resultaat_tabel(voorspellingen):
    # Elke stap op drie manieren: totaal, middelste folds samen, randfolds apart.
    rijen = []
    for model, voorspeld in voorspellingen.items():
        rijen.append(resultaat_rij(model, "Totaal", scores(voorspeld)))
        rijen.append(resultaat_rij(model, "Midden", scores_midden(voorspeld)))
        for lengte in RAND:
            rijen.append(resultaat_rij(model, f"Rand {lengte}", scores(voorspeld, masker_lengtes([lengte]))))
    return md_tabel(rijen, ["Model", "Deel", "Macro-F1", "Violations", "Violation rate",
                            "Gemiste kansen", "Gemiste-kansrate"])


def matrix_tabel(voorspeld):
    matrix = scores(voorspeld)["matrix"]
    rijen = [[label] + [int(matrix.loc[label, k]) for k in LABELS] for label in LABELS]
    return md_tabel(rijen, ["Echt \\ voorspeld"] + LABELS)

## Beslisregel en bewerking

- **Beslisregel:** een stap blijft als de macro-F1 op de middelste folds meer dan 0,005 stijgt en de violation rate niet meer dan 1 procentpunt stijgt. Of als de macro-F1 ongeveer gelijk blijft (hoogstens 0,005 verschil) en de violation rate daalt.
- **Bewerking:** alles zit in `maak_features`, en die zit in de pipeline. De schaling wordt daardoor per fold alleen op de trainfolds geleerd.

In [5]:
def beoordeel(nieuw, oud):
    verschil_f1 = nieuw["macro_f1"] - oud["macro_f1"]
    verschil_violation = nieuw["violation_rate"] - oud["violation_rate"]

    if verschil_f1 > MARGE_F1 and verschil_violation <= MARGE_VIOLATION:
        return True
    if abs(verschil_f1) <= MARGE_F1 and verschil_violation < 0:
        return True
    return False


def beste_van(kandidaten):
    # Hoogste macro-F1 op de middelste folds, bij gelijke F1 de laagste violation rate.
    return max(kandidaten, key=lambda k: (k["midden"]["macro_f1"], -k["midden"]["violation_rate"]))


def kies(kandidaten, oud):
    # Uit de kandidaten die door de beslisregel komen de beste. Komt er geen door,
    # dan de beste kandidaat, om die te kunnen laten zien. Hij blijft dan niet.
    goed = [k for k in kandidaten if beoordeel(k["midden"], oud)]
    if goed:
        return beste_van(goed), True
    return beste_van(kandidaten), False


def besluit_tekst(naam, nieuw, oud, blijft):
    return (
        f"- {naam}: **{'blijft' if blijft else 'blijft niet'}**. Middelste folds: macro-F1 "
        f"{getal(oud['macro_f1'])} naar {getal(nieuw['macro_f1'])}, violation rate "
        f"{procent(oud['violation_rate'])} naar {procent(nieuw['violation_rate'])}."
    )


def maak_features(x, budget_groot=False, log10=False, budget_per_doc=False,
                  latency_per_token=False, cpu_interacties=False):
    uit = x[["input_length_tokens", "min_macro_f1", "max_latency_ms_per_doc",
             "monthly_volume", "max_energy_kwh_per_month"]].astype(float)

    # Een lege budgetcel: 0 (zoals in de eerste notebooks) of een groot getal.
    uit["max_energy_kwh_per_month"] = uit["max_energy_kwh_per_month"].fillna(GROOT if budget_groot else 0)

    if budget_per_doc:
        budget_j = x["max_energy_kwh_per_month"] * J_PER_KWH / x["monthly_volume"]
        uit["budget_j_per_doc"] = budget_j.fillna(GROOT)
        uit["log10_budget_j_per_doc"] = np.log10(uit["budget_j_per_doc"])

    if latency_per_token:
        uit["latency_per_token"] = x["max_latency_ms_per_doc"] / x["input_length_tokens"]
        uit["log10_latency_per_token"] = np.log10(uit["latency_per_token"])

    if log10:
        for kolom in ["max_latency_ms_per_doc", "monthly_volume", "max_energy_kwh_per_month"]:
            uit[f"log10_{kolom}"] = np.log10(uit.pop(kolom))

    numeriek = list(uit.columns)

    uit["has_energy_budget"] = x["has_energy_budget"].astype(float)
    uit["hardware_rtx3080"] = (x["hardware"] == "rtx3080").astype(float)
    uit["hardware_ryzen5_3600x"] = (x["hardware"] == "ryzen5_3600x").astype(float)

    if cpu_interacties:
        for kolom in numeriek:
            uit[f"{kolom}_x_cpu"] = uit[kolom] * uit["hardware_ryzen5_3600x"]

    return uit

## Modellen en cross-validatie

`cv_voorspel` traint per fold op de andere folds. Zo wordt elk trainscenario één keer voorspeld.

In [6]:
LR = "Logistische regressie"
BOOM = "Beslisboom"
RF = "Random Forest"
HGB = "HistGradientBoosting"


def maak_schatter(soort, params):
    if soort == LR:
        return LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, **params)
    if soort == BOOM:
        return DecisionTreeClassifier(random_state=RANDOM_STATE, **params)
    if soort == RF:
        return RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE, **params)
    return HistGradientBoostingClassifier(max_iter=200, random_state=RANDOM_STATE, **params)


def maak_pipeline(soort, opties, params):
    stappen = [("features", FunctionTransformer(maak_features, kw_args=opties))]
    # Alleen de logistische regressie heeft schaling nodig.
    if soort == LR:
        stappen.append(("schaal", StandardScaler()))
    stappen.append(("model", maak_schatter(soort, params)))
    return Pipeline(stappen)


def cv_kansen(maak_model, doel):
    # Geeft per scenario de kansen, voorspeld door een model dat die fold niet zag.
    delen = []
    for fold in manifest["cv_folds"]:
        in_fold = train["family_id"].isin(fold["families"])
        model = maak_model().fit(x_ruw[~in_fold], doel[~in_fold])
        delen.append(pd.DataFrame(
            model.predict_proba(x_ruw[in_fold]), index=train.index[in_fold], columns=model.classes_
        ))
    return pd.concat(delen).reindex(train.index)


def cv_voorspel(soort, opties, params):
    kansen = cv_kansen(lambda: maak_pipeline(soort, opties, params), y)
    return kansen.idxmax(axis=1), kansen


# De huidige stand per model. Een stap die blijft, past dit aan.
huidig = {
    LR: {"opties": {}, "params": {"C": 1.0}},
    BOOM: {"opties": {}, "params": {"max_depth": 5, "min_samples_leaf": 1}},
}

# Eén rij per stap voor de overzichtstabel aan het eind.
overzicht = []


def voeg_toe_aan_overzicht(stap, model, midden, besluit):
    overzicht.append([stap, model, getal(midden["macro_f1"]), procent(midden["violation_rate"]), besluit])

## Stap 0: referenties

Baseline, logistische regressie en beslisboom met dezelfde instellingen als de bestaande notebooks. De scores moeten gelijk zijn aan `01_baseline.ipynb`, `02_logistic_regression.ipynb` en `03_decision_tree.ipynb`.

In [7]:
def baseline_voorspel():
    # De regel uit 01_baseline.ipynb: per fold drempels uit de andere trainfamilies.
    voorspeld = pd.Series(index=train.index, dtype=object)
    for fold in manifest["cv_folds"]:
        in_fold = train_configuraties["family_id"].isin(fold["families"])
        drempels = train_configuraties[~in_fold].groupby(["hardware", "category"])[
            ["macro_f1", "latency_ms_per_doc"]
        ].median()

        for index, rij in train[train["family_id"].isin(fold["families"])].iterrows():
            keuze = "geen"
            for categorie in CATEGORIE_VOORKEUR:
                f1, latency = drempels.loc[(rij["hardware"], categorie)]
                if rij["min_macro_f1"] <= f1 and rij["max_latency_ms_per_doc"] >= latency:
                    keuze = categorie
                    break
            voorspeld[index] = keuze
    return voorspeld


baseline_uit = baseline_voorspel()

for soort in [LR, BOOM]:
    voorspeld, kansen = cv_voorspel(soort, huidig[soort]["opties"], huidig[soort]["params"])
    huidig[soort].update(voorspeld=voorspeld, kansen=kansen, midden=scores_midden(voorspeld))

referenties = {
    "Baseline": baseline_uit,
    LR: huidig[LR]["voorspeld"],
    BOOM: huidig[BOOM]["voorspeld"],
}
blok("stap0_tabel", resultaat_tabel(referenties))

beste_referentie = beste_van([{"naam": n, "midden": scores_midden(v)} for n, v in referenties.items()])
blok("stap0_matrix", f"Beste op de middelste folds: {beste_referentie['naam']}.\n\n"
     + matrix_tabel(referenties[beste_referentie["naam"]]))
voeg_toe_aan_overzicht("0: referenties", beste_referentie["naam"], beste_referentie["midden"], "referentie")

@@stap0_tabel
| Model | Deel | Macro-F1 | Violations | Violation rate | Gemiste kansen | Gemiste-kansrate |
|---|---|---|---|---|---|---|
| Baseline | Totaal | 0,766 | 192 van 955 | 20,1% | 101 van 870 | 11,6% |
| Baseline | Midden | 0,791 | 132 van 639 | 20,7% | 34 van 543 | 6,3% |
| Baseline | Rand 32 | 0,708 | 1 van 158 | 0,6% | 64 van 221 | 29,0% |
| Baseline | Rand 512 | 0,701 | 59 van 158 | 37,3% | 3 van 106 | 2,8% |
| Logistische regressie | Totaal | 0,696 | 162 van 751 | 21,6% | 257 van 870 | 29,5% |
| Logistische regressie | Midden | 0,726 | 131 van 537 | 24,4% | 118 van 543 | 21,7% |
| Logistische regressie | Rand 32 | 0,626 | 18 van 164 | 11,0% | 71 van 221 | 32,1% |
| Logistische regressie | Rand 512 | 0,514 | 13 van 50 | 26,0% | 68 van 106 | 64,2% |
| Beslisboom | Totaal | 0,748 | 169 van 854 | 19,8% | 174 van 870 | 20,0% |
| Beslisboom | Midden | 0,750 | 121 van 563 | 21,5% | 94 van 543 | 17,3% |
| Beslisboom | Rand 32 | 0,682 | 2 van 144 | 1,4% | 78 van 221 | 35,3% |
| B

## Stap 1: opzoekregel

De labelregel op de dichtstbijzijnde gemeten inputlengte uit de trainfolds. Bij gelijke afstand de langere lengte, want die is strenger. Alleen configuraties uit de trainfolds.

In [8]:
def opzoekregel_voorspel():
    voorspeld = pd.Series(index=train.index, dtype=object)
    regels = []
    for fold in manifest["cv_folds"]:
        in_fold = train_configuraties["family_id"].isin(fold["families"])
        fold_configuraties = train_configuraties[~in_fold]
        gemeten = sorted(fold_configuraties["input_length_tokens"].unique())

        # Kleinste afstand. Bij gelijke afstand wint de langere lengte door -m.
        lengte = fold["input_length_tokens"]
        dichtstbij = min(gemeten, key=lambda m: (abs(m - lengte), -m))
        regels.append([lengte, int(dichtstbij)])

        for index, rij in train[train["family_id"].isin(fold["families"])].iterrows():
            scenario = rij.to_dict()
            scenario["input_length_tokens"] = dichtstbij
            voorspeld[index] = bepaal_label(fold_configuraties, scenario)
    return voorspeld, regels


opzoek_uit, opzoek_regels = opzoekregel_voorspel()

# Geen tabel per foldlengte: met 8 lengtes zijn de testlengtes daaruit af te leiden.
blok("stap1_tabel", resultaat_tabel({"Opzoekregel": opzoek_uit, "Baseline": baseline_uit}))
blok("stap1_matrix", matrix_tabel(opzoek_uit))
voeg_toe_aan_overzicht("1: opzoekregel", "Opzoekregel", scores_midden(opzoek_uit), "extra baseline")

@@stap1_tabel
| Model | Deel | Macro-F1 | Violations | Violation rate | Gemiste kansen | Gemiste-kansrate |
|---|---|---|---|---|---|---|
| Opzoekregel | Totaal | 0,868 | 46 van 791 | 5,8% | 110 van 870 | 12,6% |
| Opzoekregel | Midden | 0,886 | 36 van 521 | 6,9% | 43 van 543 | 7,9% |
| Opzoekregel | Rand 32 | 0,736 | 3 van 162 | 1,9% | 62 van 221 | 28,1% |
| Opzoekregel | Rand 512 | 0,907 | 7 van 108 | 6,5% | 5 van 106 | 4,7% |
| Baseline | Totaal | 0,766 | 192 van 955 | 20,1% | 101 van 870 | 11,6% |
| Baseline | Midden | 0,791 | 132 van 639 | 20,7% | 34 van 543 | 6,3% |
| Baseline | Rand 32 | 0,708 | 1 van 158 | 0,6% | 64 van 221 | 29,0% |
| Baseline | Rand 512 | 0,701 | 59 van 158 | 37,3% | 3 van 106 | 2,8% |
@@einde

@@stap1_matrix
| Echt \ voorspeld | compact | medium | groot | geen |
|---|---|---|---|---|
| compact | 326 | 6 | 0 | 28 |
| medium | 5 | 234 | 15 | 38 |
| groot | 0 | 10 | 164 | 44 |
| geen | 7 | 8 | 16 | 591 |
@@einde



## Stappen 2 tot en met 6: features

Elke stap voegt iets toe aan de huidige stand van het model. Blijft de stap niet, dan gaat het model verder met de oude stand.

In [9]:
def probeer_opties(stap, titel, nieuwe_opties, soorten):
    resultaten = {}
    besluiten = []
    kandidaten = []
    for soort in soorten:
        oud = huidig[soort]
        opties = {**oud["opties"], **nieuwe_opties}
        voorspeld, kansen = cv_voorspel(soort, opties, oud["params"])
        midden = scores_midden(voorspeld)
        blijft = beoordeel(midden, oud["midden"])

        resultaten[soort] = voorspeld
        kandidaten.append({"naam": soort, "midden": midden, "voorspeld": voorspeld, "blijft": blijft})
        besluiten.append(besluit_tekst(soort, midden, oud["midden"], blijft))
        if blijft:
            huidig[soort].update(opties=opties, voorspeld=voorspeld, kansen=kansen, midden=midden)

    beste = beste_van(kandidaten)
    blok(f"stap{stap}_tabel", resultaat_tabel(resultaten))
    blok(f"stap{stap}_matrix", f"Beste op de middelste folds: {beste['naam']}.\n\n" + matrix_tabel(beste["voorspeld"]))
    blok(f"stap{stap}_besluit", "\n".join(besluiten))
    voeg_toe_aan_overzicht(
        f"{stap}: {titel}", beste["naam"], beste["midden"],
        ", ".join(f"{k['naam']}: {'blijft' if k['blijft'] else 'blijft niet'}" for k in kandidaten),
    )

### Stap 2: lege budgetcel wordt een groot getal

In [10]:
probeer_opties(2, "lege budgetcel groot", {"budget_groot": True}, [LR, BOOM])

@@stap2_tabel
| Model | Deel | Macro-F1 | Violations | Violation rate | Gemiste kansen | Gemiste-kansrate |
|---|---|---|---|---|---|---|
| Logistische regressie | Totaal | 0,696 | 165 van 754 | 21,9% | 257 van 870 | 29,5% |
| Logistische regressie | Midden | 0,720 | 134 van 537 | 25,0% | 121 van 543 | 22,3% |
| Logistische regressie | Rand 32 | 0,641 | 17 van 164 | 10,4% | 70 van 221 | 31,7% |
| Logistische regressie | Rand 512 | 0,527 | 14 van 53 | 26,4% | 66 van 106 | 62,3% |
| Beslisboom | Totaal | 0,729 | 140 van 779 | 18,0% | 219 van 870 | 25,2% |
| Beslisboom | Midden | 0,754 | 80 van 488 | 16,4% | 128 van 543 | 23,6% |
| Beslisboom | Rand 32 | 0,593 | 7 van 138 | 5,1% | 88 van 221 | 39,8% |
| Beslisboom | Rand 512 | 0,742 | 53 van 153 | 34,6% | 3 van 106 | 2,8% |
@@einde

@@stap2_matrix
Beste op de middelste folds: Beslisboom.

| Echt \ voorspeld | compact | medium | groot | geen |
|---|---|---|---|---|
| compact | 306 | 8 | 0 | 46 |
| medium | 2 | 214 | 8 | 68 |
| groot | 0 | 

### Stap 3: log10

Van `max_latency_ms_per_doc`, `monthly_volume` en `max_energy_kwh_per_month`. Dit kan alleen met het grote getal uit stap 2, want de log10 van 0 bestaat niet. Daarom hoort `budget_groot` bij deze stap, ook als stap 2 niet bleef.

In [11]:
probeer_opties(3, "log10", {"budget_groot": True, "log10": True}, [LR, BOOM])

@@stap3_tabel
| Model | Deel | Macro-F1 | Violations | Violation rate | Gemiste kansen | Gemiste-kansrate |
|---|---|---|---|---|---|---|
| Logistische regressie | Totaal | 0,746 | 151 van 792 | 19,1% | 192 van 870 | 22,1% |
| Logistische regressie | Midden | 0,778 | 125 van 569 | 22,0% | 70 van 543 | 12,9% |
| Logistische regressie | Rand 32 | 0,668 | 15 van 168 | 8,9% | 63 van 221 | 28,5% |
| Logistische regressie | Rand 512 | 0,577 | 11 van 55 | 20,0% | 59 van 106 | 55,7% |
| Beslisboom | Totaal | 0,729 | 140 van 780 | 17,9% | 218 van 870 | 25,1% |
| Beslisboom | Midden | 0,755 | 80 van 489 | 16,4% | 127 van 543 | 23,4% |
| Beslisboom | Rand 32 | 0,593 | 7 van 138 | 5,1% | 88 van 221 | 39,8% |
| Beslisboom | Rand 512 | 0,742 | 53 van 153 | 34,6% | 3 van 106 | 2,8% |
@@einde

@@stap3_matrix
Beste op de middelste folds: Logistische regressie.

| Echt \ voorspeld | compact | medium | groot | geen |
|---|---|---|---|---|
| compact | 317 | 8 | 0 | 35 |
| medium | 11 | 181 | 3 | 97 |
| gr

### Stap 4: budget per document

In [12]:
probeer_opties(4, "budget per document", {"budget_per_doc": True}, [LR, BOOM])

@@stap4_tabel
| Model | Deel | Macro-F1 | Violations | Violation rate | Gemiste kansen | Gemiste-kansrate |
|---|---|---|---|---|---|---|
| Logistische regressie | Totaal | 0,759 | 142 van 793 | 17,9% | 180 van 870 | 20,7% |
| Logistische regressie | Midden | 0,800 | 114 van 569 | 20,0% | 59 van 543 | 10,9% |
| Logistische regressie | Rand 32 | 0,669 | 16 van 169 | 9,5% | 62 van 221 | 28,1% |
| Logistische regressie | Rand 512 | 0,562 | 12 van 55 | 21,8% | 59 van 106 | 55,7% |
| Beslisboom | Totaal | 0,790 | 134 van 844 | 15,9% | 145 van 870 | 16,7% |
| Beslisboom | Midden | 0,829 | 97 van 583 | 16,6% | 45 van 543 | 8,3% |
| Beslisboom | Rand 32 | 0,610 | 7 van 138 | 5,1% | 89 van 221 | 40,3% |
| Beslisboom | Rand 512 | 0,805 | 30 van 123 | 24,4% | 11 van 106 | 10,4% |
@@einde

@@stap4_matrix
Beste op de middelste folds: Beslisboom.

| Echt \ voorspeld | compact | medium | groot | geen |
|---|---|---|---|---|
| compact | 324 | 8 | 0 | 28 |
| medium | 2 | 245 | 4 | 41 |
| groot | 0 | 13

### Stap 5: latency per token

In [13]:
probeer_opties(5, "latency per token", {"latency_per_token": True}, [LR, BOOM])

@@stap5_tabel
| Model | Deel | Macro-F1 | Violations | Violation rate | Gemiste kansen | Gemiste-kansrate |
|---|---|---|---|---|---|---|
| Logistische regressie | Totaal | 0,740 | 133 van 759 | 17,5% | 205 van 870 | 23,6% |
| Logistische regressie | Midden | 0,789 | 94 van 526 | 17,9% | 83 van 543 | 15,3% |
| Logistische regressie | Rand 32 | 0,558 | 20 van 151 | 13,2% | 83 van 221 | 37,6% |
| Logistische regressie | Rand 512 | 0,701 | 19 van 82 | 23,2% | 39 van 106 | 36,8% |
| Beslisboom | Totaal | 0,829 | 119 van 875 | 13,6% | 98 van 870 | 11,3% |
| Beslisboom | Midden | 0,860 | 80 van 582 | 13,7% | 29 van 543 | 5,3% |
| Beslisboom | Rand 32 | 0,675 | 10 van 163 | 6,1% | 67 van 221 | 30,3% |
| Beslisboom | Rand 512 | 0,831 | 29 van 130 | 22,3% | 2 van 106 | 1,9% |
@@einde

@@stap5_matrix
Beste op de middelste folds: Beslisboom.

| Echt \ voorspeld | compact | medium | groot | geen |
|---|---|---|---|---|
| compact | 337 | 9 | 0 | 14 |
| medium | 2 | 253 | 9 | 28 |
| groot | 0 | 13 |

### Stap 6: hardware-interacties (alleen logistische regressie)

In [14]:
probeer_opties(6, "hardware-interacties", {"cpu_interacties": True}, [LR])

@@stap6_tabel
| Model | Deel | Macro-F1 | Violations | Violation rate | Gemiste kansen | Gemiste-kansrate |
|---|---|---|---|---|---|---|
| Logistische regressie | Totaal | 0,776 | 128 van 800 | 16,0% | 165 van 870 | 19,0% |
| Logistische regressie | Midden | 0,823 | 103 van 572 | 18,0% | 49 van 543 | 9,0% |
| Logistische regressie | Rand 32 | 0,672 | 17 van 173 | 9,8% | 59 van 221 | 26,7% |
| Logistische regressie | Rand 512 | 0,570 | 8 van 55 | 14,5% | 57 van 106 | 53,8% |
@@einde

@@stap6_matrix
Beste op de middelste folds: Logistische regressie.

| Echt \ voorspeld | compact | medium | groot | geen |
|---|---|---|---|---|
| compact | 318 | 11 | 0 | 31 |
| medium | 9 | 199 | 5 | 79 |
| groot | 0 | 24 | 139 | 55 |
| geen | 37 | 18 | 40 | 527 |
@@einde

@@stap6_besluit
- Logistische regressie: **blijft**. Middelste folds: macro-F1 0,800 naar 0,823, violation rate 20,0% naar 18,0%.
@@einde



## Stap 7: hyperparameters

Kiezen op de middelste folds. Dat is iets optimistisch, want de score komt van dezelfde folds als de keuze.

In [15]:
def probeer_grid(soort, grid, oud_midden, opties):
    kandidaten = []
    for params in grid:
        voorspeld, kansen = cv_voorspel(soort, opties, params)
        kandidaten.append({"naam": soort, "params": params, "midden": scores_midden(voorspeld),
                           "voorspeld": voorspeld, "kansen": kansen})
    winnaar, blijft = kies(kandidaten, oud_midden)
    return winnaar, blijft, kandidaten


def grid_tabel(kandidaten, aantal):
    volgorde = sorted(kandidaten, key=lambda k: (-k["midden"]["macro_f1"], k["midden"]["violation_rate"]))
    rijen = [
        [", ".join(f"{k}={v}" for k, v in c["params"].items()), getal(c["midden"]["macro_f1"]),
         procent(c["midden"]["violation_rate"]), procent(c["midden"]["gemiste_kans_rate"])]
        for c in volgorde[:aantal]
    ]
    return md_tabel(rijen, ["Instellingen", "Macro-F1 midden", "Violation rate midden", "Gemiste-kansrate midden"])


grids = {
    LR: [{"C": c} for c in [0.01, 0.1, 1, 10, 100]],
    BOOM: [{"max_depth": d, "min_samples_leaf": b} for d in range(3, 11) for b in [1, 5, 20]],
}

resultaten, besluiten, winnaars = {}, [], []
for soort, grid in grids.items():
    oud_midden = huidig[soort]["midden"]
    winnaar, blijft, kandidaten = probeer_grid(soort, grid, oud_midden, huidig[soort]["opties"])

    blok(f"stap7_grid_{'lr' if soort == LR else 'boom'}", grid_tabel(kandidaten, 5))
    resultaten[soort] = winnaar["voorspeld"]
    winnaars.append({**winnaar, "blijft": blijft})
    besluiten.append(besluit_tekst(f"{soort} ({winnaar['params']})", winnaar["midden"], oud_midden, blijft))
    if blijft:
        huidig[soort].update(params=winnaar["params"], voorspeld=winnaar["voorspeld"],
                             kansen=winnaar["kansen"], midden=winnaar["midden"])

beste = beste_van(winnaars)
blok("stap7_tabel", resultaat_tabel(resultaten))
blok("stap7_matrix", f"Beste op de middelste folds: {beste['naam']}.\n\n" + matrix_tabel(beste["voorspeld"]))
blok("stap7_besluit", "\n".join(besluiten))
voeg_toe_aan_overzicht(
    "7: hyperparameters", beste["naam"], beste["midden"],
    ", ".join(f"{k['naam']}: {'blijft' if k['blijft'] else 'blijft niet'}" for k in winnaars),
)

# De stand per model na stap 7.
blok("stap7_stand", "\n".join(
    f"- {soort}: opties {huidig[soort]['opties']}, instellingen {huidig[soort]['params']}." for soort in huidig
))

@@stap7_grid_lr
| Instellingen | Macro-F1 midden | Violation rate midden | Gemiste-kansrate midden |
|---|---|---|---|
| C=100 | 0,843 | 16,4% | 6,4% |
| C=10 | 0,840 | 17,4% | 6,6% |
| C=1 | 0,823 | 18,0% | 9,0% |
| C=0.1 | 0,785 | 21,1% | 12,7% |
| C=0.01 | 0,708 | 24,6% | 20,8% |
@@einde



@@stap7_grid_boom
| Instellingen | Macro-F1 midden | Violation rate midden | Gemiste-kansrate midden |
|---|---|---|---|
| max_depth=7, min_samples_leaf=5 | 0,919 | 5,6% | 5,3% |
| max_depth=9, min_samples_leaf=5 | 0,919 | 5,6% | 5,3% |
| max_depth=10, min_samples_leaf=5 | 0,919 | 5,6% | 5,3% |
| max_depth=8, min_samples_leaf=5 | 0,918 | 5,8% | 5,3% |
| max_depth=7, min_samples_leaf=1 | 0,918 | 5,9% | 5,0% |
@@einde

@@stap7_tabel
| Model | Deel | Macro-F1 | Violations | Violation rate | Gemiste kansen | Gemiste-kansrate |
|---|---|---|---|---|---|---|
| Logistische regressie | Totaal | 0,782 | 123 van 806 | 15,3% | 149 van 870 | 17,1% |
| Logistische regressie | Midden | 0,843 | 95 van 579 | 16,4% | 35 van 543 | 6,4% |
| Logistische regressie | Rand 32 | 0,648 | 21 van 170 | 12,4% | 60 van 221 | 27,1% |
| Logistische regressie | Rand 512 | 0,555 | 7 van 57 | 12,3% | 54 van 106 | 50,9% |
| Beslisboom | Totaal | 0,893 | 53 van 833 | 6,4% | 76 van 870 | 8,7% |
| Beslisboom | Midden | 0,9

## Stap 8: drempel op de kansen

Voor het beste model tot nu toe. Een categorie wordt alleen gekozen als de kans op die categorie minstens `t` is, anders wordt het `geen`.

In [16]:
soort_beste = beste_van([{"naam": s, "midden": huidig[s]["midden"]} for s in huidig])["naam"]
beste_tot_nu = {"naam": soort_beste, "voorspeld": huidig[soort_beste]["voorspeld"],
                "midden": huidig[soort_beste]["midden"]}
blok("stap8_model", f"Beste model tot nu toe: {soort_beste}.")


def pas_drempel_toe(voorspeld, kansen, t):
    # De kans van de voorspelde categorie, per scenario.
    kans = pd.Series(kansen.to_numpy()[np.arange(len(kansen)), kansen.columns.get_indexer(voorspeld)],
                     index=voorspeld.index)
    uit = voorspeld.copy()
    uit[(voorspeld != "geen") & (kans < t)] = "geen"
    return uit


def drempel_tabel(kandidaten):
    rijen = []
    for k in kandidaten:
        for deel, s in k["delen"].items():
            rijen.append([k["naam"], getal(k["t"], 1), deel, getal(s["macro_f1"]),
                          procent(s["violation_rate"]), procent(s["gemiste_kans_rate"])])
    return md_tabel(rijen, ["Model", "t", "Deel", "Macro-F1", "Violation rate", "Gemiste-kansrate"])


def drempel_kandidaat(naam, t, voorspeld):
    return {
        "naam": naam, "t": t, "voorspeld": voorspeld, "midden": scores_midden(voorspeld),
        "delen": {
            "Totaal": scores(voorspeld),
            "Midden": scores_midden(voorspeld),
            **{f"Rand {lengte}": scores(voorspeld, masker_lengtes([lengte])) for lengte in RAND},
        },
    }


kandidaten = [
    drempel_kandidaat(soort_beste, t, pas_drempel_toe(huidig[soort_beste]["voorspeld"], huidig[soort_beste]["kansen"], t))
    for t in DREMPELS
]
winnaar, blijft = kies(kandidaten, beste_tot_nu["midden"])

blok("stap8_drempels", drempel_tabel(kandidaten))
blok("stap8_tabel", resultaat_tabel({f"{soort_beste}, t={getal(winnaar['t'], 1)}": winnaar["voorspeld"]}))
blok("stap8_matrix", f"Beste drempel: t={getal(winnaar['t'], 1)}.\n\n" + matrix_tabel(winnaar["voorspeld"]))
blok("stap8_besluit", besluit_tekst(f"{soort_beste} met t={getal(winnaar['t'], 1)}", winnaar["midden"],
                                    beste_tot_nu["midden"], blijft))
voeg_toe_aan_overzicht("8: drempel op kansen", f"{soort_beste}, t={getal(winnaar['t'], 1)}", winnaar["midden"],
                       "blijft" if blijft else "blijft niet")

if blijft:
    beste_tot_nu = {"naam": f"{soort_beste}, t={getal(winnaar['t'], 1)}", "voorspeld": winnaar["voorspeld"],
                    "midden": winnaar["midden"]}

@@stap8_model
Beste model tot nu toe: Beslisboom.
@@einde



@@stap8_drempels
| Model | t | Deel | Macro-F1 | Violation rate | Gemiste-kansrate |
|---|---|---|---|---|---|
| Beslisboom | 0,5 | Totaal | 0,893 | 6,4% | 8,7% |
| Beslisboom | 0,5 | Midden | 0,919 | 5,6% | 5,3% |
| Beslisboom | 0,5 | Rand 32 | 0,776 | 3,9% | 19,9% |
| Beslisboom | 0,5 | Rand 512 | 0,887 | 13,7% | 2,8% |
| Beslisboom | 0,6 | Totaal | 0,892 | 6,2% | 9,2% |
| Beslisboom | 0,6 | Midden | 0,919 | 5,4% | 5,5% |
| Beslisboom | 0,6 | Rand 32 | 0,776 | 3,9% | 19,9% |
| Beslisboom | 0,6 | Rand 512 | 0,877 | 13,3% | 5,7% |
| Beslisboom | 0,7 | Totaal | 0,885 | 4,7% | 12,5% |
| Beslisboom | 0,7 | Midden | 0,914 | 4,3% | 8,1% |
| Beslisboom | 0,7 | Rand 32 | 0,762 | 2,3% | 23,5% |
| Beslisboom | 0,7 | Rand 512 | 0,872 | 10,8% | 12,3% |
| Beslisboom | 0,8 | Totaal | 0,884 | 4,7% | 12,8% |
| Beslisboom | 0,8 | Midden | 0,912 | 4,3% | 8,5% |
| Beslisboom | 0,8 | Rand 32 | 0,762 | 2,3% | 23,5% |
| Beslisboom | 0,8 | Rand 512 | 0,872 | 10,8% | 12,3% |
| Beslisboom | 0,9 | Totaal | 0,8

## Stap 9: Random Forest en HistGradientBoosting

Met de features van de beslisboom, want dit zijn ook boommodellen. Kleine grid, gekozen op de middelste folds.

In [17]:
boom_opties = huidig[BOOM]["opties"]

grids_9 = {
    RF: [{"max_depth": d, "min_samples_leaf": b} for d in [5, 8, None] for b in [1, 5, 20]],
    HGB: [{"max_depth": d, "min_samples_leaf": b, "learning_rate": l}
          for d in [3, 5, 8] for b in [5, 20] for l in [0.05, 0.1]],
}

resultaten, besluiten, winnaars, beste_params = {}, [], [], {}
for soort, grid in grids_9.items():
    winnaar, blijft, kandidaten = probeer_grid(soort, grid, beste_tot_nu["midden"], boom_opties)

    blok(f"stap9_grid_{'rf' if soort == RF else 'hgb'}", grid_tabel(kandidaten, 5))
    resultaten[soort] = winnaar["voorspeld"]
    beste_params[soort] = winnaar["params"]
    winnaars.append({**winnaar, "blijft": blijft})
    besluiten.append(besluit_tekst(f"{soort} ({winnaar['params']})", winnaar["midden"], beste_tot_nu["midden"], blijft))

beste = beste_van(winnaars)
blok("stap9_tabel", resultaat_tabel(resultaten))
blok("stap9_matrix", f"Beste op de middelste folds: {beste['naam']}.\n\n" + matrix_tabel(beste["voorspeld"]))
blok("stap9_besluit", "\n".join(besluiten))
voeg_toe_aan_overzicht(
    "9: Random Forest en HGB", beste["naam"], beste["midden"],
    ", ".join(f"{k['naam']}: {'blijft' if k['blijft'] else 'blijft niet'}" for k in winnaars),
)

# Alleen het beste model dat door de beslisregel kwam, wordt de nieuwe maat.
geslaagd = [k for k in winnaars if k["blijft"]]
if geslaagd:
    nieuw = beste_van(geslaagd)
    beste_tot_nu = {"naam": nieuw["naam"], "voorspeld": nieuw["voorspeld"], "midden": nieuw["midden"]}

@@stap9_grid_rf
| Instellingen | Macro-F1 midden | Violation rate midden | Gemiste-kansrate midden |
|---|---|---|---|
| max_depth=None, min_samples_leaf=1 | 0,918 | 6,5% | 4,6% |
| max_depth=8, min_samples_leaf=1 | 0,910 | 7,5% | 5,0% |
| max_depth=8, min_samples_leaf=5 | 0,907 | 7,8% | 4,8% |
| max_depth=None, min_samples_leaf=5 | 0,906 | 7,9% | 5,0% |
| max_depth=5, min_samples_leaf=5 | 0,892 | 8,1% | 7,9% |
@@einde



@@stap9_grid_hgb
| Instellingen | Macro-F1 midden | Violation rate midden | Gemiste-kansrate midden |
|---|---|---|---|
| max_depth=8, min_samples_leaf=5, learning_rate=0.05 | 0,932 | 5,5% | 3,3% |
| max_depth=8, min_samples_leaf=5, learning_rate=0.1 | 0,929 | 6,2% | 3,1% |
| max_depth=5, min_samples_leaf=20, learning_rate=0.1 | 0,929 | 5,8% | 3,1% |
| max_depth=5, min_samples_leaf=5, learning_rate=0.05 | 0,927 | 6,2% | 2,9% |
| max_depth=8, min_samples_leaf=20, learning_rate=0.1 | 0,927 | 6,1% | 3,7% |
@@einde

@@stap9_tabel
| Model | Deel | Macro-F1 | Violations | Violation rate | Gemiste kansen | Gemiste-kansrate |
|---|---|---|---|---|---|---|
| Random Forest | Totaal | 0,891 | 56 van 835 | 6,7% | 77 van 870 | 8,9% |
| Random Forest | Midden | 0,918 | 35 van 542 | 6,5% | 25 van 543 | 4,6% |
| Random Forest | Rand 32 | 0,743 | 9 van 179 | 5,0% | 50 van 221 | 22,6% |
| Random Forest | Rand 512 | 0,907 | 12 van 114 | 10,5% | 2 van 106 | 1,9% |
| HistGradientBoosting | Totaal | 0,910 |

## Stap 10: twee ja/nee-modellen

Eén model per categorie: "compact voldoet", "medium voldoet" en "groot voldoet". De targets komen uit `voldoende_categorieen`. Dat zijn labels, geen features. Keuze: de kleinste categorie met een kans van minstens t, anders `geen`.

Bij HistGradientBoosting houdt `monotonic_cst` de richting vast: een strengere eis mag de kans nooit verhogen.

In [18]:
doelen = {c: voldoende.map(lambda v, c=c: c in v) for c in CATEGORIE_VOORKEUR}


def monotone_richting(kolom):
    if kolom == "has_energy_budget" or kolom.startswith("hardware"):
        return 0
    if "min_macro_f1" in kolom or "input_length_tokens" in kolom or "monthly_volume" in kolom:
        return -1
    if "latency" in kolom or "budget" in kolom or "energy" in kolom:
        return 1
    return 0


def maak_lr_ja_nee():
    opties, params = huidig[LR]["opties"], huidig[LR]["params"]
    return maak_pipeline(LR, opties, params)


def maak_hgb_ja_nee():
    kolommen = maak_features(x_ruw.head(), **boom_opties).columns
    params = {**beste_params[HGB], "monotonic_cst": {k: monotone_richting(k) for k in kolommen}}
    return maak_pipeline(HGB, boom_opties, params)


blok("stap10_richting", md_tabel(
    [[k, monotone_richting(k)] for k in maak_features(x_ruw.head(), **boom_opties).columns],
    ["Kolom", "Richting"],
))


def kies_cascade(kansen, t):
    # Van groot naar klein invullen, zodat de kleinste categorie met kans >= t wint.
    uit = pd.Series("geen", index=train.index, dtype=object)
    for categorie in reversed(CATEGORIE_VOORKEUR):
        uit[kansen[categorie] >= t] = categorie
    return uit


kandidaten = []
for naam, maak in {f"Ja/nee {LR}": maak_lr_ja_nee, f"Ja/nee {HGB} monotoon": maak_hgb_ja_nee}.items():
    kansen = {c: cv_kansen(maak, doelen[c])[True] for c in CATEGORIE_VOORKEUR}
    for t in DREMPELS:
        kandidaten.append(drempel_kandidaat(naam, t, kies_cascade(kansen, t)))

winnaar, blijft = kies(kandidaten, beste_tot_nu["midden"])
blok("stap10_drempels", drempel_tabel(kandidaten))
naam_winnaar = f"{winnaar['naam']}, t={getal(winnaar['t'], 1)}"
blok("stap10_tabel", resultaat_tabel({naam_winnaar: winnaar["voorspeld"]}))
blok("stap10_matrix", f"Beste op de middelste folds: {naam_winnaar}.\n\n" + matrix_tabel(winnaar["voorspeld"]))
blok("stap10_besluit", besluit_tekst(naam_winnaar, winnaar["midden"], beste_tot_nu["midden"], blijft))
voeg_toe_aan_overzicht("10: twee ja/nee-modellen", naam_winnaar, winnaar["midden"], "blijft" if blijft else "blijft niet")

if blijft:
    beste_tot_nu = {"naam": naam_winnaar, "voorspeld": winnaar["voorspeld"], "midden": winnaar["midden"]}

@@stap10_richting
| Kolom | Richting |
|---|---|
| input_length_tokens | -1 |
| min_macro_f1 | -1 |
| budget_j_per_doc | 1 |
| log10_budget_j_per_doc | 1 |
| latency_per_token | 1 |
| log10_latency_per_token | 1 |
| log10_max_latency_ms_per_doc | 1 |
| log10_monthly_volume | -1 |
| log10_max_energy_kwh_per_month | 1 |
| has_energy_budget | 0 |
| hardware_rtx3080 | 0 |
| hardware_ryzen5_3600x | 0 |
@@einde



@@stap10_drempels
| Model | t | Deel | Macro-F1 | Violation rate | Gemiste-kansrate |
|---|---|---|---|---|---|
| Ja/nee Logistische regressie | 0,5 | Totaal | 0,733 | 17,2% | 20,0% |
| Ja/nee Logistische regressie | 0,5 | Midden | 0,804 | 17,3% | 9,0% |
| Ja/nee Logistische regressie | 0,5 | Rand 32 | 0,568 | 18,4% | 30,8% |
| Ja/nee Logistische regressie | 0,5 | Rand 512 | 0,519 | 12,5% | 53,8% |
| Ja/nee Logistische regressie | 0,6 | Totaal | 0,712 | 13,8% | 24,4% |
| Ja/nee Logistische regressie | 0,6 | Midden | 0,778 | 13,9% | 13,1% |
| Ja/nee Logistische regressie | 0,6 | Rand 32 | 0,572 | 14,5% | 34,8% |
| Ja/nee Logistische regressie | 0,6 | Rand 512 | 0,464 | 10,6% | 60,4% |
| Ja/nee Logistische regressie | 0,7 | Totaal | 0,680 | 11,2% | 29,5% |
| Ja/nee Logistische regressie | 0,7 | Midden | 0,741 | 10,9% | 18,6% |
| Ja/nee Logistische regressie | 0,7 | Rand 32 | 0,552 | 12,3% | 37,6% |
| Ja/nee Logistische regressie | 0,7 | Rand 512 | 0,423 | 10,8% | 68,9% |
| Ja/nee Logisti

## Stap 11: conservatieve opzoekregel

Net als de opzoekregel uit stap 1, maar voorzichtiger. Twee varianten:

- **Variant a, langere buur:** de labelregel op de kleinste gemeten trainlengte die groter is dan de lengte van de fold.
- **Variant b, beide buren:** een categorie voldoet alleen als hij op de dichtstbijzijnde kortere én langere gemeten trainlengte voldoet. Daarna dezelfde voorkeur als de labelregel.

Alleen configuraties uit de trainfolds. Is er geen langere trainlengte (fold 512), dan valt de regel terug op de dichtstbijzijnde lengte, zoals in stap 1. Variant b heeft bij fold 64 geen kortere trainlengte. Daar telt alleen de langere buur.

In [19]:
from src.candidate.data.labelling_rule import CATEGORIE_VOORKEUR, LABEL_GEEN

TERUGVAL = "terugval: geen langere lengte"
ALLEEN_LANGER = "alleen langere buur: geen kortere lengte"


def dichtstbijzijnde(lengte, gemeten):
    # Dezelfde keuze als in stap 1: kleinste afstand, bij gelijke afstand de langere lengte.
    return min(gemeten, key=lambda m: (abs(m - lengte), -m))


def langere_buur(lengte, gemeten):
    langer = [m for m in gemeten if m > lengte]
    if not langer:
        return [dichtstbijzijnde(lengte, gemeten)], TERUGVAL
    return [min(langer)], ""


def beide_buren(lengte, gemeten):
    korter = [m for m in gemeten if m < lengte]
    langer = [m for m in gemeten if m > lengte]
    if not langer:
        return [dichtstbijzijnde(lengte, gemeten)], TERUGVAL
    if not korter:
        return [min(langer)], ALLEEN_LANGER
    return [max(korter), min(langer)], ""


def conservatieve_voorspel(kies_buren):
    # Dezelfde opzet als opzoekregel_voorspel in stap 1. Alleen de trainfolds tellen mee.
    voorspeld = pd.Series(index=train.index, dtype=object)
    regels = {}
    for fold in manifest["cv_folds"]:
        in_fold = train_configuraties["family_id"].isin(fold["families"])
        fold_configuraties = train_configuraties[~in_fold]
        gemeten = sorted(fold_configuraties["input_length_tokens"].unique())

        lengte = fold["input_length_tokens"]
        buren, opmerking = kies_buren(lengte, gemeten)
        tekst = " en ".join(str(int(b)) for b in buren)
        regels[lengte] = f"{tekst} ({opmerking})" if opmerking else tekst

        for index, rij in train[train["family_id"].isin(fold["families"])].iterrows():
            scenario = rij.to_dict()

            # Een categorie moet op elke opgezochte lengte voldoen.
            per_buur = [
                voldoende_categorieen(fold_configuraties, {**scenario, "input_length_tokens": buur})
                for buur in buren
            ]
            samen = set.intersection(*per_buur)

            # Dezelfde voorkeur als de labelregel: compact vóór medium, anders geen.
            voorspeld[index] = next((c for c in CATEGORIE_VOORKEUR if c in samen), LABEL_GEEN)
    return voorspeld, regels


VARIANT_A = "Variant a: langere buur"
VARIANT_B = "Variant b: beide buren"

langer_uit, langer_regels = conservatieve_voorspel(langere_buur)
beide_uit, beide_regels = conservatieve_voorspel(beide_buren)

# Geen tabel per foldlengte: met 8 lengtes zijn de testlengtes daaruit af te leiden.

blok("stap11_tabel", resultaat_tabel({
    VARIANT_A: langer_uit,
    VARIANT_B: beide_uit,
    "Opzoekregel (stap 1)": opzoek_uit,
    f"Eindmodel stap 10: {beste_tot_nu['naam']}": beste_tot_nu["voorspeld"],
}))

beste = beste_van([
    {"naam": VARIANT_A, "midden": scores_midden(langer_uit), "voorspeld": langer_uit},
    {"naam": VARIANT_B, "midden": scores_midden(beide_uit), "voorspeld": beide_uit},
])
blok("stap11_matrix", f"Beste variant op de middelste folds: {beste['naam']}.\n\n" + matrix_tabel(beste["voorspeld"]))
voeg_toe_aan_overzicht("11: conservatieve opzoekregel", beste["naam"], beste["midden"], "extra baseline")

@@stap11_tabel
| Model | Deel | Macro-F1 | Violations | Violation rate | Gemiste kansen | Gemiste-kansrate |
|---|---|---|---|---|---|---|
| Variant a: langere buur | Totaal | 0,876 | 24 van 757 | 3,2% | 126 van 870 | 14,5% |
| Variant a: langere buur | Midden | 0,900 | 14 van 487 | 2,9% | 59 van 543 | 10,9% |
| Variant a: langere buur | Rand 32 | 0,736 | 3 van 162 | 1,9% | 62 van 221 | 28,1% |
| Variant a: langere buur | Rand 512 | 0,907 | 7 van 108 | 6,5% | 5 van 106 | 4,7% |
| Variant b: beide buren | Totaal | 0,875 | 15 van 750 | 2,0% | 131 van 870 | 15,1% |
| Variant b: beide buren | Midden | 0,898 | 5 van 480 | 1,0% | 64 van 543 | 11,8% |
| Variant b: beide buren | Rand 32 | 0,736 | 3 van 162 | 1,9% | 62 van 221 | 28,1% |
| Variant b: beide buren | Rand 512 | 0,907 | 7 van 108 | 6,5% | 5 van 106 | 4,7% |
| Opzoekregel (stap 1) | Totaal | 0,868 | 46 van 791 | 5,8% | 110 van 870 | 12,6% |
| Opzoekregel (stap 1) | Midden | 0,886 | 36 van 521 | 6,9% | 43 van 543 | 7,9% |
| Opzoekrege

## Overzicht

In [20]:
blok("overzicht", md_tabel(overzicht, ["Stap", "Beste model in de stap", "Macro-F1 midden",
                                         "Violation rate midden", "Besluit"]))

# Het beste model aan het eind, naast de opzoekregel en de baseline.
blok("eind_tabel", resultaat_tabel({
    beste_tot_nu["naam"]: beste_tot_nu["voorspeld"],
    "Opzoekregel": opzoek_uit,
    "Baseline": baseline_uit,
}))

@@overzicht
| Stap | Beste model in de stap | Macro-F1 midden | Violation rate midden | Besluit |
|---|---|---|---|---|
| 0: referenties | Baseline | 0,791 | 20,7% | referentie |
| 1: opzoekregel | Opzoekregel | 0,886 | 6,9% | extra baseline |
| 2: lege budgetcel groot | Beslisboom | 0,754 | 16,4% | Logistische regressie: blijft niet, Beslisboom: blijft |
| 3: log10 | Logistische regressie | 0,778 | 22,0% | Logistische regressie: blijft, Beslisboom: blijft |
| 4: budget per document | Beslisboom | 0,829 | 16,6% | Logistische regressie: blijft, Beslisboom: blijft |
| 5: latency per token | Beslisboom | 0,860 | 13,7% | Logistische regressie: blijft niet, Beslisboom: blijft |
| 6: hardware-interacties | Logistische regressie | 0,823 | 18,0% | Logistische regressie: blijft |
| 7: hyperparameters | Beslisboom | 0,919 | 5,6% | Logistische regressie: blijft, Beslisboom: blijft |
| 8: drempel op kansen | Beslisboom, t=0,6 | 0,919 | 5,4% | blijft |
| 9: Random Forest en HGB | HistGradientBoosti